# Export the workspace with its results
The zip below saves the **code**. Everything the pipeline produced lives in Snowflake (tables, the staged PDFs, AI results, evaluations), so the cell before it first writes those into the workspace under `results/<YYYY-MM-DD>/`. The zip then contains both.

1. **Save results** (run first): one CSV per table and result view, the staged PDFs, and `SUMMARY.md` (scope, latest batch, pass counts, data-quality checks, row count of every file). VARIANT/ARRAY/OBJECT columns are written as JSON text. No AI calls.
2. **Export zip**: zips the whole workspace, including `results/`, to `@lab_export_stage`.

`MAKE_SNAPSHOT = True` also creates a zero-copy clone of the schema (`ATP524_SNAPSHOT_<date>`) as an in-Snowflake backup. Whether a clone copies stage files is not verified here, so the PDF download in step 1 is the reliable copy of the PDFs.

In [ ]:
import csv
import datetime as dt
import json
import shutil
from pathlib import Path

from snowflake.snowpark.context import get_active_session

MAKE_SNAPSHOT = False   # True: also CREATE SCHEMA ATP524_SNAPSHOT_<date> CLONE ATP524 (zero-copy backup)

session = get_active_session()
for statement in ("USE DATABASE BOOTCAMP", "USE SCHEMA ATP524", "USE WAREHOUSE COMPUTE_WH"):
    session.sql(statement).collect()

stamp = dt.date.today().isoformat()
out = Path.cwd() / "results" / stamp
(out / "pdfs").mkdir(parents=True, exist_ok=True)

# Every lab table plus the result views people read. Views are evaluated now, so the CSV
# records what they showed at export time.
OBJECTS = [
    "lab_settings", "metric_catalog", "statement_heading_patterns",          # CONFIG
    "sec_facts", "ingest_log", "statement_page_scan", "parse_results",       # RAW
    "extraction_results", "extracted_values", "synthetic_ground_truth",
    "report_manifest", "lab_runs",                                           # CURATED
    "report_readiness", "pipeline_coverage", "metric_evaluation",            # ANALYSIS (views)
]

def cell_text(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, default=str)
    return str(value)

row_counts = {}
for name in OBJECTS:
    df = session.table(name)
    rows = df.collect()
    with open(out / f"{name}.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(df.columns)
        for row in rows:
            writer.writerow([cell_text(v) for v in row])
    row_counts[name] = len(rows)
    print(f"  {name:26s} {len(rows):6d} rows")

# The staged files (annual reports + synthetic batch), one by one under their stage path
# (e.g. pdfs/AAPL/2025.pdf), so same-named files in different folders never collide.
# Streaming avoids the temporary .part files the workspace filesystem does not allow.
# Every file is checked against the stage's size afterwards; any gap stops the export.
staged = session.sql("SELECT relative_path, size FROM DIRECTORY(@annual_report_pdfs) ORDER BY 1").collect()
for r in staged:
    target = out / "pdfs" / r["RELATIVE_PATH"]
    target.parent.mkdir(parents=True, exist_ok=True)
    with session.file.get_stream(f"@annual_report_pdfs/{r['RELATIVE_PATH']}") as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)
problems = [f"{r['RELATIVE_PATH']} (stage {r['SIZE']} bytes, saved "
            f"{(out / 'pdfs' / r['RELATIVE_PATH']).stat().st_size if (out / 'pdfs' / r['RELATIVE_PATH']).exists() else 'missing'})"
            for r in staged
            if not (out / "pdfs" / r["RELATIVE_PATH"]).exists()
            or (out / "pdfs" / r["RELATIVE_PATH"]).stat().st_size != r["SIZE"]]
if problems:
    raise RuntimeError("Staged files not saved correctly:\n  " + "\n  ".join(problems))
print(f"  {'@annual_report_pdfs':26s} {len(staged):6d} files (all sizes verified)")

# SUMMARY.md is organized by annual report (ticker + year), not by scope or latest batch:
# every report in the export appears once with its LATEST run, so nothing is hidden.
# lab_settings only says what the next run will process; years in that scope that are not
# ingested yet are listed too, so no year is silently missing.
LATEST_RUN = """latest AS (
    SELECT * FROM lab_runs
    QUALIFY ROW_NUMBER() OVER (PARTITION BY relative_path ORDER BY created_at DESC) = 1)"""
reports = session.sql(f"""
    WITH {LATEST_RUN},
    scores AS (
        SELECT run_id,
               COUNT_IF(method = 'DIRECT_PDF' AND evaluation_status = 'PASS') AS direct_pass,
               COUNT_IF(method = 'DIRECT_PDF') AS direct_n,
               COUNT_IF(method = 'PARSE_THEN_EXTRACT' AND evaluation_status = 'PASS') AS parse_pass,
               COUNT_IF(method = 'PARSE_THEN_EXTRACT') AS parse_n
        FROM metric_evaluation GROUP BY run_id),
    scope_years AS (
        SELECT s.ticker, s.first_year + g.n AS report_year
        FROM lab_settings s,
             (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => 10))) g
        WHERE s.first_year + g.n <= s.last_year)
    SELECT COALESCE(m.ticker, y.ticker) AS ticker, COALESCE(m.report_year, y.report_year) AS report_year,
           m.page_ranges[0]:start::INT AS page_start, m.page_ranges[0]:"end"::INT AS page_end,
           COALESCE(m.locator_method, IFF(m.approved, 'before locator_method', NULL)) AS locator,
           CASE WHEN m.relative_path IS NULL THEN 'Not ingested (in next-run scope)'
                WHEN NOT COALESCE(m.approved, FALSE) THEN 'Rejected by 05 (see review_notes)'
                WHEN l.run_id IS NULL THEN 'Ready, not extracted'
                WHEN sc.run_id IS NULL THEN 'Extracted, not evaluated'
                ELSE 'Evaluated' END AS status,
           l.created_at::DATE AS latest_run, l.metric_count,
           sc.direct_pass, sc.direct_n, sc.parse_pass, sc.parse_n
    FROM report_manifest m
    FULL OUTER JOIN scope_years y ON y.ticker = m.ticker AND y.report_year = m.report_year
    LEFT JOIN latest l ON l.relative_path = m.relative_path
    LEFT JOIN scores sc ON sc.run_id = l.run_id
    ORDER BY 1, 2""").collect()
failures = session.sql(f"""
    WITH {LATEST_RUN}
    SELECT e.ticker, e.report_year, e.method, e.metric_id, e.evaluation_status,
           e.raw_value::STRING AS raw_value, e.sec_value
    FROM metric_evaluation e JOIN latest l ON l.run_id = e.run_id
    WHERE e.evaluation_status <> 'PASS'
    ORDER BY 1, 2, 3, 4""").collect()
batches = session.sql("""
    SELECT r.batch_id, MIN(r.created_at) AS started,
           LISTAGG(DISTINCT r.ticker, ', ') WITHIN GROUP (ORDER BY r.ticker) AS tickers,
           COUNT(DISTINCT r.run_id) AS reports, MIN(r.metric_count) AS min_metrics, MAX(r.metric_count) AS max_metrics,
           COUNT_IF(e.evaluation_status = 'PASS') AS passed, COUNT(e.run_id) AS expected
    FROM lab_runs r LEFT JOIN metric_evaluation e ON e.run_id = r.run_id
    GROUP BY r.batch_id ORDER BY started""").collect()
scope = session.sql("SELECT ticker, first_year, last_year FROM lab_settings").collect()[0]

def score(passed, total):
    return f"{passed}/{total}" if total else "-"

def blank(value):
    return "-" if value is None else value

companies = sorted({r["TICKER"] for r in reports})
annual_reports = [r for r in reports if r["STATUS"] != "Not ingested (in next-run scope)"]
lines = [f"# Lab results exported {stamp}", "",
         f"Contains: {len(companies)} companies ({', '.join(companies)}), {len(annual_reports)} annual reports, "
         f"{row_counts['lab_runs']} extraction runs in {len(batches)} batches, {len(staged)} staged PDFs.  ",
         f"Next run is set to: {scope['TICKER']} {scope['FIRST_YEAR']}-{scope['LAST_YEAR']} "
         "(`lab_settings`; change it in step 00). This does not filter anything below.", "",
         "## 1. Every annual report: status and latest result", "",
         "Each report's most recent extraction run. Scores are metrics passing SEC ground truth / metrics asked.", "",
         "| Ticker | Year | Pages | Located by | Status | Latest run | Metrics | Direct PDF | Parse then extract |",
         "| --- | --- | --- | --- | --- | --- | --- | --- | --- |"]
for r in reports:
    pages = f"[{r['PAGE_START']}, {r['PAGE_END']})" if r["PAGE_START"] is not None else "-"
    lines.append(f"| {r['TICKER']} | {r['REPORT_YEAR']} | {pages} | {blank(r['LOCATOR'])} | {r['STATUS']} | "
                 f"{blank(r['LATEST_RUN'])} | {blank(r['METRIC_COUNT'])} | {score(r['DIRECT_PASS'], r['DIRECT_N'])} | "
                 f"{score(r['PARSE_PASS'], r['PARSE_N'])} |")
lines += ["", "## 2. Not passing (latest runs only)", ""]
if failures:
    lines += ["| Ticker | Year | Method | Metric | Status | AI returned | SEC value |", "| --- | --- | --- | --- | --- | --- | --- |"]
    lines += [f"| {r['TICKER']} | {r['REPORT_YEAR']} | {r['METHOD']} | {r['METRIC_ID']} | {r['EVALUATION_STATUS']} | "
              f"`{blank(r['RAW_VALUE'])}` | {blank(r['SEC_VALUE'])} |" for r in failures]
else:
    lines.append("Every metric in every report's latest run passed.")
lines += ["", "## 3. Run history (one row per 08 batch, oldest first)", "",
          "Includes earlier runs that section 1 replaces with a newer run (e.g. the 3-metric AAPL runs).", "",
          "| Started | Batch | Companies | Reports | Metrics | Passed / expected |", "| --- | --- | --- | --- | --- | --- |"]
for r in batches:
    metrics = str(r["MIN_METRICS"]) if r["MIN_METRICS"] == r["MAX_METRICS"] else f"{r['MIN_METRICS']}-{r['MAX_METRICS']}"
    lines.append(f"| {r['STARTED']:%Y-%m-%d %H:%M} | `{r['BATCH_ID'][:8]}` | {r['TICKERS']} | {r['REPORTS']} | "
                 f"{metrics} | {score(r['PASSED'], r['EXPECTED'])} |")
lines += ["", "## 4. Files in this export", "", "| File | Rows |", "| --- | --- |"]
lines += [f"| {name}.csv | {count} |" for name, count in row_counts.items()]
lines += [f"| pdfs/ | {len(staged)} files (sizes verified against the stage) |", "",
          "Data-quality checks: run sql/10.validation.sql (or the runner's step 10 cell) on the live schema."]
(out / "SUMMARY.md").write_text("\n".join(lines) + "\n")

if MAKE_SNAPSHOT:
    snap = f"ATP524_SNAPSHOT_{stamp.replace('-', '')}"
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {snap} CLONE ATP524").collect()
    print(f"Snapshot schema created: BOOTCAMP.{snap}")

print(f"\nResults saved to {out.relative_to(Path.cwd())}; now run the export zip cell.")

In [ ]:
import shutil
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# 1. Set your database and schema explicitly
session.sql('USE DATABASE BOOTCAMP').collect()
session.sql('USE SCHEMA ATP524').collect()

# 2. Zip the entire workspace folder into /tmp/lab_export.zip
shutil.make_archive('/tmp/lab_export', 'zip', '.')

# 3. Create the stage if it doesn't exist and upload the zip file
session.sql('CREATE STAGE IF NOT EXISTS lab_export_stage').collect()
session.file.put('/tmp/lab_export.zip', '@lab_export_stage', auto_compress=False, overwrite=True)

print("Success! Download lab_export.zip from BOOTCAMP -> ATP524 -> Stages -> LAB_EXPORT_STAGE")

